In [17]:
import pandas as pd
import geopandas as gpd
import os
import numpy as np
import pyarrow.parquet as pq
from shapely.geometry import LineString, MultiLineString


import matplotlib.pyplot as plt
import seaborn as sns
import calendar
import shapely as wkt
import ast
import re

import folium  # For plots of geometry on interactive map
import matplotlib.colors as mcolors
import random

import plotly.express as px
import plotly.io as pio

pio.renderers.default = "colab"  # or: "notebook_connected"
from plotly.graph_objs import Font

import sys
from pathlib import Path

DATA_DIR = Path.cwd().parent / "data"
print("Data directory:", DATA_DIR.resolve())

Data directory: C:\Users\cort3\Documents\Classes\OptimalChargerPlacement\data


In [57]:
MONTH = 6
HOUR = 20

Need a list of census blocks in Alameda county
Get list of intersecting feeders from metatadata of all CA feeders
Get Feeder data for only feeders intersectiing alameda county census blocks



### CENSUS BLOCK DATA FROM THIBAUD - Get census block geodataframe for alameda county  



import census block geodataframe from  shp file  
filter to only alameda county census blocks

In [18]:
# Import Census tract data
def import_census_block_gdf_for_charging(base_path=DATA_DIR):
    relative_folder_path = "Geography_Files"
    filename = "location_str_to_geoid_mapping.shp"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    gdf = gpd.read_file(file_path)
    return gdf

def get_census_block_gdf_almada_charging():
    """
    Get census block geodataframe for Alameda county
    """
    df = import_census_block_gdf_for_charging()
    df = df[df["GEOID_STR"].str.contains("Alameda")]
    return df

alamedaCensusGdf = get_census_block_gdf_almada_charging()



### Grid Metadata and Geometry
Raw data file is for the entire state of California.
ADD FILTER TO ONLY INCLUDE ALAMEDA COUNTY

In [19]:
# Import metadata geodataframe for all feeders
# 1 row for each feeder
# 'feeder_id'	'division'	'substation'	'nom_volt_kV'	'Existing_DG'	'Queued_DG'	'shape_length'	'geometry'


def import_metaData_gdf_all_feeders(base_path=DATA_DIR):

    relative_folder_path = "cleanedGridData"
    filename = "feeder_meta_clean_gdf.gpkg"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    gdf_all_feeders = gpd.read_file(file_path)
    return gdf_all_feeders


feederMetadataGdf = import_metaData_gdf_all_feeders()
display(feederMetadataGdf.head())

,feeder_id,division,substation,nom_volt_kV,Existing_DG,Queued_DG,shape_length,geometry
0,163201102,Stockton,WEST POINT,12kV,2340,370,431704.873501,"MULTILINESTRING ((717038.22 4245680.843, 71703..."
1,083432111,San Jose,HICKS,21kV,4720,730,61473.923814,"MULTILINESTRING ((600661.102 4121410.616, 6006..."
2,022101109,Peninsula,SF H,12kV,1520,80,25925.744324,"MULTILINESTRING ((551452.358 4173482.555, 5513..."
3,043302103,Sonoma,MONROE,21kV,10210,1040,109445.067948,"MULTILINESTRING ((522459.857 4258793.177, 5224..."
4,162671103,Yosemite,WESTLEY,12kV,1870,5440,161530.404081,"MULTILINESTRING ((652837.489 4156520.205, 6527..."


In [20]:
# Find all feeders in feederMetadataGdf located in San Francisco (divission == "San Francisco")
sanFranciscoFeeders = feederMetadataGdf[
    feederMetadataGdf["division"] == "San Francisco"
]["feeder_id"].tolist()
print(sanFranciscoFeeders)

['022011118', '022801111', '022801128', '022031110', '022331102', '022510401', '022011162', '022801113', '022530403', '022331107', '022510402', '022480402', '022261103', '022801136', '022090402', '022780403', '022801132', '022801127', '022090409', '022130408', '022070405', '022801140', '022260402', '022031111', '022011128', '022011104', '022270406', '022071101', '022011107', '022031113', '022011111', '022270405', '022781102', '022091101', '022070401', '022090410', '022011160', '022031104', '022801124', '022801110', '022871112', '022270402', '022011110', '022260405', '022871121', '022680401', '022011113', '022580401', '022801108', '022280408', '022801137', '022261102', '022090401', '022270404', '022801135', '022781103', '022871119', '022090405', '022011124', '022873401', '022610402', '022260407', '022801122', '022070407', '022031112', '022801105', '022011127', '022011105', '022220409', '022873405', '022801104', '022011106', '022490401', '022071103', '022801109', '022801131', '022390401'

### Create intersection matrix, find all feeders that intersect Alameda county census blocks

In [32]:
def get_feeder_census_block_intersection_matrix(feederGdf, censusGdf, verbose=False):
    # CRS for Bay Area can be set to 32610 (when working in meters)
    # CRS for some parts of California need to be adjusted
    blocks_proj_m = alamedaCensusGdf.to_crs(epsg=32610)
    feeders_proj_m = feederMetadataGdf.to_crs(epsg=32610)

    # spatial join to find intersections between feeders and census blocks
    intersections = gpd.overlay(feeders_proj_m, blocks_proj_m, how="intersection")
    
    # length of each feeder segment in each block
    intersections["length_in_block_m"] = intersections.geometry.apply(get_length)

    # total length per block
    block_totals = (
        intersections.groupby("GEOID")["length_in_block_m"]
        .sum()
        .rename("total_length_in_block"))
    
    # merge total length per block back to intersections
    intersections = intersections.merge(block_totals, on="GEOID")

    # Fraction of a feeder lenth in a block over total length of all feeders in that block
    intersections["fraction"] = (intersections["length_in_block_m"] / intersections["total_length_in_block"])

    feeder_block_matrix = intersections.pivot_table(
        index="feeder_id",  # replace with your feeder ID column
        columns="GEOID",
        values="fraction",
        fill_value=0,)
    
    if verbose:
        display(feeder_block_matrix)
        # CHECKING THE INTERSECTION WORKED CORRECTLY
        # Add row to feeder_block_matrix which is the sum of each column
        court_row = pd.DataFrame(feeder_block_matrix.sum(axis=0)).T
        court_row.index = ["Court_Sum"]
        vals = feeder_block_matrix.sum(axis=0).unique()
        np.set_printoptions(precision=4)
        print(f"All block cols sum to {vals}")
        
    return feeder_block_matrix

def get_length(geom):
    if geom.is_empty:
        return 0
    elif isinstance(geom, LineString):
        return geom.length
    elif isinstance(geom, MultiLineString):
        return sum(line.length for line in geom.geoms)
    else:
        # GeometryCollection: sum only LineStrings
        return sum(g.length for g in geom.geoms if isinstance(g, LineString))

feederBlockMtxAlameda = get_feeder_census_block_intersection_matrix(feederMetadataGdf, alamedaCensusGdf, verbose=True)

GEOID,060014001001,060014001002,060014002001,060014002002,060014003001,060014003002,060014003003,060014003004,060014004001,060014004002,...,060014517031,060014517032,060014517041,060014517042,060014517043,060019819001,060019820001,060019821001,060019832001,060019832002
feeder_id,,,,,,,,,,,,,,,,,,,,,
012011101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.027052,0.000000
012011102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.003189,0.0,0.075951,0.003121
012011103,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.003623,0.0,0.024106,0.000000
012011104,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.059989,0.000000
012011105,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.066367,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
014722111,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
082831109,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000
163741102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.000000,0.000000


All block cols sum to [1. 1. 1. 1.]


In [52]:
# get list of all feeders from the feeder_id in feederBlockMtxAlameda

alamedaFeedersList = feederBlockMtxAlameda.index.tolist()
# Convert list of strings to list of integers
alamedaFeedersAsIntsList = [int(feeder) for feeder in alamedaFeedersList]

In [64]:
# Save feederBlockMtxAlameda to csv
output_filename = "feederBlockIntersectionMtx.csv"
output_filepath = os.path.join(DATA_DIR, "OptimizationInput", output_filename)
feederBlockMtxAlameda.to_csv(output_filepath, index=False)

### EV Projection Data for Census Blocks

In [22]:
# HELPER FUNCTIONS FOR EV LOAD CURVES

def import_ev_projection_df(year, type="unmanaged"):
    """
    Import raw ev projection csv as dataframe.
    Limited to Alameda county and 2025 or 2035 projections
    Given one day of 15 min data for load curves as a string, convert the string to list of floats
    """
    if type not in ["managed", "unmanaged", "all"]:
        raise ValueError("type must be 'managed', 'unmanaged', or 'all'")
    if year not in [ "full_ev_adoption_alameda_load_curves.csv", "2035", "2025"]:
        raise ValueError("year must be 'full_ev_adoption_alameda_load_curves.csv', '2035', or '2025'")

    if type == "unmanaged":
        if year == "full_addoption":
            rel_file_path = "full_ev_adoption_alameda_load_curves.csv"
        elif year == "2035":
            rel_file_path = "sim_20251117_alameda_2035_adoption/ev_load_curves.csv"
        elif year == "2025":
            rel_file_path = "sim_20251117_alameda_2025_adoption/ev_load_curves.csv"

    elif type == "managed":
        if year == "full_addoption":
            rel_file_path = "full_ev_adoption_alameda_load_curves.csv"
        elif year == "2035":
            rel_file_path = "sim_20251121_alameda_2035_managed_naive/ev_load_curves.csv"
        elif year == "2025":
            rel_file_path = "sim_20251121_alameda_2025_managed_naive/ev_load_curves.csv"

    print(year)
    file_path = os.path.join(DATA_DIR, "EvChargingLoadCurves", rel_file_path)
    df = pd.read_csv(file_path)
    
    # Convert string "[...]" → Python list of floats
    if "load_curve" in df.columns:
        df["load_curve"] = df["load_curve"].apply(
            lambda x: ast.literal_eval(x) if isinstance(x, str) else x
        )
    return df


# ev_projection_df = import_ev_projection_df("2035", type="managed")
# display(ev_projection_df.head())


def expand_load_curve_to_hourly(df, method="mean"):
    """
    Expands one day of 15 min load curves (96 points) into 24 hourly columns.

    INPUT: 
    df - from import_ev_projection_df() with a column named 'load_curve' containing list of 96 floats
    method ("mean" or "max") - how to aggregate 4 15 min values into one hour value

    RETURN: 
    df - Replaces the list of 15 min load data in input df with 24 columns for hourly load
    """

    if method not in ["mean", "max"]:
        raise ValueError("method must be 'mean' or 'max'")

    # Function to aggregate one row’s curve into 24 hourly values
    def aggregate_curve(curve):
        curve = np.array(curve)
        hourly = []
        for h in range(24):
            block = curve[h * 4 : (h + 1) * 4]
            if method == "mean":
                hourly.append(block.mean())
            else:  # method == "max"
                hourly.append(block.max())
        return hourly

    # Expand into a DataFrame
    hourly_expanded = df["load_curve"].apply(aggregate_curve)
    hourly_df = pd.DataFrame(
        hourly_expanded.tolist(), columns=[f"{i}" for i in range(24)]
    )

    # Return original df + new columns
    return pd.concat([df.drop(columns=["load_curve"]), hourly_df], axis=1)

# hrly = expand_load_curve_to_hourly(ev_projection_df, method="mean")
# display(hrly.head())
# # list all unique values in charger type column
# print(ev_projection_df["charger_type"].unique())

def sum_EVloads_in_blockGrp_for_type(df, charger_type="all"):
    """Sums ev loads in each block group by charger type (public, private, all). 
    Ensures every geoid appears in output."""
    public_types = [
        "Public_HD_Long_Duration_DCFC_150_kW", "Public_HD_Long_Duration_DCFC_350_kW",
        "Public_MD_Long_Duration_DCFC_150_kW", "Public_MD_Long_Duration_DCFC_50_kW",
        "Public_MD_Long_Duration_L2_11_kW", "Public_MD_Opportunistic_DCFC_350_kW",
        "Public_MD_Opportunistic_DCFC_50_kW", "Public_LD_L2_11_KW",
        "Public_LD_DCFC_150_kW", "Public_LD_DCFC_250_kW",
        "Public_LD_DCFC_350_kW", "Public_HD_Long_Duration_DCFC_50_kW",
        "Public_HD_Long_Duration_DCFC_750_kW", "Public_HD_Opportunistic_DCFC_350_kW",
        "Public_MD_Opportunistic_DCFC_1000_kW", "Public_HD_Opportunistic_DCFC_50_kW",
        "Public_HD_Opportunistic_DCFC_1000_kW",
    ]

    private_types = ["SFH_LD_L2", "Work_LD_L2", "MFH_LD_L2"]

    if charger_type == "all":
        filtered_df = df.copy()
    elif charger_type == "public":
        filtered_df = df[df["charger_type"].isin(public_types)].copy()
    else:  # private
        filtered_df = df[df["charger_type"].isin(private_types)].copy()

    hourly_cols = [str(i) for i in range(24)]

    # --- CORE FIX: ensure every geoid appears ---
    all_geoids = df["geoid"].unique()

    summed_df = (
        filtered_df.groupby("geoid")[hourly_cols]
        .sum()
        .reindex(all_geoids, fill_value=0)  # <--- THIS LINE FIXES THE PROBLEM
        .reset_index()
    )

    return summed_df
# summed_loads_public = sum_EVloads_in_blockGrp_for_type(hrly, charger_type="public")
# display(summed_loads_public.head())
# summed_loads_private = sum_EVloads_in_blockGrp_for_type(hrly, charger_type="private")
# display(summed_loads_private.head())
# summed_loads_all = sum_EVloads_in_blockGrp_for_type(hrly, charger_type="all")
# display(summed_loads_all.head())



In [23]:
def get_clean_EV_load_df(year='2025', chargerType='all', scenario="unmanaged", method="mean"):
    """
    Returns a cleaned EV load DataFrame 
    from file for a given year, charger type, scenario. 
    Selects method for aggregating 15 min load data to hour (mean or sum).
    """
    EvProjDf = import_ev_projection_df(year, scenario)
    EvProjDf = expand_load_curve_to_hourly(EvProjDf, method=method)
    EvProjDf = sum_EVloads_in_blockGrp_for_type(EvProjDf, charger_type=chargerType)
    return EvProjDf
evLoad2035Df = get_clean_EV_load_df(year='2035', chargerType='all', scenario="unmanaged", method="mean")
display(evLoad2035Df.head())

2035


,geoid,0,1,2,3,4,5,6,7,8,...,14,15,16,17,18,19,20,21,22,23
0,"0 (Tract 9900, Alameda, CA)",11.2,11.2,11.2,11.2,11.2,11.200000,11.2,2.8,0.00,...,911.400001,518.7,61.950000,46.20,46.200,46.2,46.2,46.2,46.2,25.839984
1,"0 (Tract 9901, San Mateo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,236.25,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,0.0,0.0,0.000000
2,"1 (Tract 1, Carson City, NV)",22.4,22.4,22.4,22.4,22.4,10.960008,0.0,0.0,0.00,...,0.000000,0.0,0.000000,0.00,0.000,0.0,0.0,5.6,22.4,22.400000
3,"1 (Tract 1, Fresno, CA)",18.4,18.4,24.0,29.6,29.6,29.600000,24.2,22.4,22.40,...,11.200000,16.6,11.600006,7.20,7.200,7.2,7.2,7.2,7.2,18.400000
4,"1 (Tract 1, Inyo, CA)",0.0,0.0,0.0,0.0,0.0,0.000000,0.0,0.0,0.00,...,0.000000,0.0,0.000000,68.25,34.125,0.0,0.0,0.0,0.0,0.000000


In [24]:
def get_clean_EV_growth_df(year1='2025', chargerType1='all', scenario1="unmanaged", year2='2035', chargerType2="all", scenario2="managed", method="mean"):
    """
    1. Gets hourly EV load data for two years each with specific scenarios, and charger types
    2. Computes growth from year1 to year2 for each block group given
    """
    yr1EvProjDf = import_ev_projection_df(year1, scenario1)
    yr1EvProjDf = expand_load_curve_to_hourly(yr1EvProjDf, method=method)
    yr1EvProjDf = sum_EVloads_in_blockGrp_for_type(yr1EvProjDf, charger_type=chargerType1)

    yr2EvProjDf = import_ev_projection_df(year2, scenario2)
    yr2EvProjDf = expand_load_curve_to_hourly(yr2EvProjDf, method=method)
    yr2EvProjDf = sum_EVloads_in_blockGrp_for_type(yr2EvProjDf, charger_type=chargerType2)

    # # Make a new dataframe and Subtract the hourly values of year 1 from year 2 to get growth
    # growth_df = yr2EvProjDf.copy()
    # hourly_cols = [str(i) for i in range(24)]
    # # growth_df[hourly_cols] = yr2EvProjDf[hourly_cols] - yr1EvProjDf[hourly_cols]

    hourly_cols = [str(i) for i in range(24)]

    # --- 1. Check same geoids ---
    geoids_2025 = set(yr1EvProjDf["geoid"])
    geoids_2035 = set(yr2EvProjDf["geoid"])

    missing_in_2035 = geoids_2025 - geoids_2035
    missing_in_2025 = geoids_2035 - geoids_2025

    if missing_in_2035 or missing_in_2025:
        raise ValueError(
            f"Geoids do not match.\n"
            f"Missing in 2035: {missing_in_2035}\n"
            f"Missing in 2025: {missing_in_2025}"
        )

    # --- 2. Sort & align by geoid ---
    df2025_sorted = yr1EvProjDf.sort_values("geoid").reset_index(drop=True)
    df2035_sorted = yr2EvProjDf.sort_values("geoid").reset_index(drop=True)

    # --- 3. Compute growth (2035 - 2025) ---
    growth_matrix = df2035_sorted[hourly_cols].values - df2025_sorted[hourly_cols].values

    growth_df = pd.DataFrame(growth_matrix, columns=hourly_cols)
    growth_df.insert(0, "geoid", df2025_sorted["geoid"].values)

    return growth_df
    return growth_df
ev_growth_df = get_clean_EV_growth_df(year1='2035', chargerType1='all', scenario1="unmanaged", year2='2035', chargerType2="all", scenario2="unmanaged", method="mean")

2035
2035


### Grid Data

#### Base Load Data

In [61]:
# base load data for all feeders in Alameda
# first get base load data for all feeders
# Then filter to only Alameda feeders using alamedaFeedersList


# LOAD DATA ALL FEEDERS
def import_loadDf_all_feeders(base_path=DATA_DIR):

    relative_folder_path = "cleanedGridData"
    filename = "feeder_load_profile_clean.csv"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    load_df_all_feeders = pd.read_csv(file_path)

    # load_df_all_feeders = pd.read_csv(file_path, dtype={'feeder_id': str})
    return load_df_all_feeders


def get_base_load_df_for_feeders_in_list(feedersList):
    """
    Get load dataframe for only Alameda county feeders
    """
    load_df_all_feeders = import_loadDf_all_feeders()
    filteredDf = load_df_all_feeders[
        load_df_all_feeders["feeder_id"].isin(feedersList)
    ].copy()
    return filteredDf

def get_base_load_df_for_timeslice_and_type(df, month, hour, low_or_high='high'):

    filtered_df = df[(df["month"] == month) & (df["hour"] == hour)].copy()
    filtered_df = filtered_df.drop(columns=["month", "hour"])
    if low_or_high == 'high':
        # drop low column
        filtered_df = filtered_df.drop(columns=["low_load_kw"])
    elif low_or_high == 'low':
        # drop high column
        filtered_df = filtered_df.drop(columns=["high_load_kw"])
    else:
        raise ValueError("low_or_high must be 'low' or 'high'")

    return filtered_df


alamedaFeedersBaseLoadDf = get_base_load_df_for_feeders_in_list(alamedaFeedersAsIntsList)
# display(alamedaFeedersBaseLoadDf.head())

alamedaBaseLoadDf_slice = get_base_load_df_for_timeslice_and_type(alamedaFeedersBaseLoadDf, MONTH, HOUR, low_or_high='high')
display(alamedaBaseLoadDf_slice.head())

# Save alamedaBaseLoadDf_slice as alamedaBaseLoadDf_slice_month{MONTH}_hour{HOUR}.csv
output_filename = f"alamedaBaseLoadDf_slice_month{MONTH}_hour{HOUR}.csv"
output_filepath = os.path.join(DATA_DIR, "OptimizationInput", output_filename)
alamedaBaseLoadDf_slice.to_csv(output_filepath, index=False)

,feeder_id,high_load_kw
36428,12011109,4855.0
36716,12011114,4454.0
37004,12011115,3393.0
37292,12011116,2380.0
37580,12011133,4971.0


#### ICA DATA
import ICA load cap data for given set of feeders, only keep specified columns, or only keep min ICA data
import a single feeders ICA data only keeping specified columns

RAW LINE PARQUET TO FEEDER AGGREGATION
import raw parquet data for a given feeders lines, only keep specific columns
aggregate line data to get feeder level statistics


FOR ALL FEEDERS IN A LIST (ALAMEDA)
Get df of mine ICA with repect to time for all feeders in alameda county
Then just take the one specific time slice for all feeders in alameda county





In [26]:
# Import ICA data from parquet files
def get_FeederMinIcaDf_from_parquet(feeder_id, verbose=True, base_path=DATA_DIR, ):
    """
    get_feeder_ICA_df_from_parquet, remove some columns, and aggregat all IC data to min IC per line month-hour

    :feeder_id: Feeder ID whose ICA data you want
    :folder_path: Path to the folder in Google Drive that holds the .parquet files
    :return: Pandas DataFrame
    """
    # Drop division column as it is already in metadata for each feeder

    relative_folder_path = "cleanedGridData/ICA_Load_CLEAN_PARQUET_v4"
    filename = f"{feeder_id}.parquet"
    file_path = os.path.join(base_path, relative_folder_path, filename)
    missingFeeder = None

    # If file does not exist, return dataframe with nan values for all columns except feeder_id
    if not os.path.exists(file_path):
        missingFeeder = feeder_id  
        if verbose:
            print(f"File not found: {feeder_id}")

        # create dataframe from cols_to_keep use given feeder_id and nan for all other columns
        cols = ["feeder_id", "month", "hour", "IC_min_kW"]
        df = pd.DataFrame(columns=cols)
        
        # Set up df with all month-hour combinations with NaN IC_min_kW
        df = pd.DataFrame([(feeder_id, month, hour, np.nan) for month in range(1, 13) for hour in range(24)], columns=cols)
        
        if verbose:
            display(df.head())

        return df, missingFeeder

    cols_to_keep = [
        "line_section_id", "month", "hour", "feeder_id",
        "IC10_Thermal_KW", "IC10_Voltage_KW",
        "IC90_Thermal_KW", "IC90_Voltage_KW"
    ]    
    table = pq.read_table(file_path, columns=cols_to_keep)
    df = table.to_pandas()

    # Combine IC columns into single IC_min column
    ic_cols = [
        "IC10_Thermal_KW", "IC10_Voltage_KW",
        "IC90_Thermal_KW", "IC90_Voltage_KW"
    ]

    df["IC_min_kW"] = df[ic_cols].min(axis=1)
    df = df.drop(columns=ic_cols)

    # Aggregate line data to feeder-month-hour level by taking mean of IC_min for all lines in feeder
    agg_df = df.groupby(["feeder_id", "month", "hour"], as_index=False).agg({"IC_min_kW": "mean"})

    # only return missingFeeder if it is defined

    return agg_df, missingFeeder

(df, missingFeeder) = get_FeederMinIcaDf_from_parquet("163201102")
display(df.head())

,feeder_id,month,hour,IC_min_kW
0,163201102,1,0,669.080933
1,163201102,1,1,699.204956
2,163201102,1,2,711.028442
3,163201102,1,3,710.940918
4,163201102,1,4,710.430359


In [ ]:
def get_allMinICA_df_for_feeders(feeder_id_list, base_path=DATA_DIR):
    """
    For a list of feeder IDs, get their min ICA data DataFrames and concatenate into one DataFrame.

    :feeder_ids: List of feeder IDs
    :base_path: Base path to data directory
    :return: Concatenated Pandas DataFrame of min ICA data for all feeders
    """
    df_list = []
    missingFeederList = []
    for feeder_id in feeder_id_list:
        (df, missing_feeder) = get_FeederMinIcaDf_from_parquet(feeder_id, verbose=False, base_path=base_path)
        if missing_feeder is not None:
            missingFeederList.append(missing_feeder)
        df_list.append(df)

    combined_df = pd.concat(df_list, ignore_index=True)

    print(f"Missing feeders: {missingFeederList}")
    return combined_df
alamedaMinICA = get_allMinICA_df_for_feeders(alamedaFeedersList)


Missing feeders: ['012011101', '012011102', '012011103', '012011104', '012011105', '012011106', '012041104', '012041112', '012060406', '012111111', '012111112', '012111113', '012111114', '012111115', '012111116', '013681111', '014722102']


C:\Users\cort3\AppData\Local\Temp\ipykernel_11796\3342249324.py:17: FutureWarning:

The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.



In [63]:
def get_minICA_df_time_slice(df, month, hour):
    """
    Given a DataFrame of min ICA data for multiple feeders,
    return a DataFrame with min ICA values at a specific month-hour time slice.

    :df: DataFrame with columns ['feeder_id', 'month', 'hour', 'IC_min_kW']
    :month: Month (1-12) to filter on
    :hour: Hour (0-23) to filter on
    :return: Filtered DataFrame with min ICA values at the specified time slice
    """
    # if a feeder has a nan value for IC_min_kW at the given month-hour, it should still appear in output with nan value
    filtered_df = df[(df['month'] == month) & (df['hour'] == hour)].copy()
    
    # Rename IC_min_kW column to indicate the time slice
    # filtered_df = filtered_df.rename(columns={'IC_min_kW': f'IC_min_kW_mon{month}_hr{hour}'})
    
    # drop month and hour columns
    filtered_df = filtered_df.drop(columns=['month', 'hour'])
    return filtered_df

df_minICA_Alameda_slice = get_minICA_df_time_slice(alamedaMinICA, month=MONTH, hour=HOUR)
display(df_minICA_Alameda_slice.head())

output_filename = f"alamedaMinICA_slice_month{MONTH}_hour{HOUR}.csv"
output_filepath = os.path.join(DATA_DIR, "OptimizationInput", output_filename)
df_minICA_Alameda_slice.to_csv(output_filepath, index=False)

,feeder_id,IC_min_kW
140,012011101,NaN
428,012011102,NaN
716,012011103,NaN
1004,012011104,NaN
1292,012011105,NaN
